# MI4344 — Benchmark Nhân Ma Trận trên GPU (CUDA Naive vs cuBLAS)

- **Đề tài:** Thiết kế, mô hình hóa và đánh giá hiệu năng ba kiến trúc tính toán (CPU, GPU và Systolic Array) trên bài toán nhân ma trận $C = A \times B$ (`float32`).
- **Thành viên phụ trách:** **P3 — Nguyễn Hoàng Long** (MSSV: `202418938`, Nhóm 15).
- **Câu hỏi nghiên cứu 2:** *Khi nào GPU thực sự vượt trội hơn CPU? Điểm hòa vốn kích thước ma trận $N^*$ là bao nhiêu khi tính toán thuần ($T_{\text{kernel}}$) và khi tính toàn trình cả thời gian truyền dữ liệu qua PCIe ($T_{\text{E2E}}$)?*

### Cơ sở lý thuyết (Patterson & Hennessy, *Computer Organization and Design*, 5th Ed.)
1. **Kiến trúc SIMT & Latency Hiding (Ch. 6):** GPU sở hữu hàng nghìn luồng nhẹ (threads) tổ chức theo Warp trên các Streaming Multiprocessors (SM), che giấu độ trễ bộ nhớ bằng chuyển ngữ cảnh phần cứng tức thì thay vì bộ nhớ đệm lớn như CPU.
2. **Tháp bộ nhớ & Nút thắt PCIe (Ch. 5–6):** Khi thực thi $C = A \times B$ trên GPU rời qua bus PCIe, toàn trình gồm 3 pha đo bằng `cudaEvent_t`:
   $$T_{\text{E2E}} = T_{\text{H2D}} + T_{\text{kernel}} + T_{\text{D2H}}$$
3. **Cường độ số học qua PCIe (Roofline Model):** Tổng phép tính là $2N^3$ FLOPs, tổng dữ liệu truyền qua PCIe ($A, B$ đi vào và $C$ đi ra, mỗi phần tử 4 byte `float32`) là $12N^2$ bytes $\implies I_{\text{PCIe}}(N) = \frac{2N^3}{12N^2} = \frac{N}{6}$ FLOPs/Byte. Khi $N$ nhỏ, $I_{\text{PCIe}}$ thấp và chi phí cố định (kernel launch, copy PCIe) làm GPU **chậm hơn CPU**.

In [ ]:
import datetime
import json
import os
import platform
import re
import subprocess

def run_cmd(cmd):
    try:
        return subprocess.check_output(cmd, shell=True, stderr=subprocess.STDOUT, text=True).strip()
    except subprocess.CalledProcessError as e:
        return e.output.strip()

print("=== 1. Kiểm tra GPU & CUDA trên Colab ===")
print(run_cmd("nvidia-smi"))
print("\n--- nvcc --version ---")
nvcc_out = run_cmd("nvcc --version")
print(nvcc_out)

gpu_name = run_cmd("nvidia-smi --query-gpu=name --format=csv,noheader | head -n 1") or "Unknown GPU"
vram_mb = run_cmd("nvidia-smi --query-gpu=memory.total --format=csv,noheader,nounits | head -n 1")
driver_ver = run_cmd("nvidia-smi --query-gpu=driver_version --format=csv,noheader | head -n 1")
cap_raw = run_cmd("nvidia-smi --query-gpu=compute_cap --format=csv,noheader | head -n 1")

cuda_match = re.search(r"release\s+([0-9]+\.[0-9]+)", nvcc_out)
cuda_ver = cuda_match.group(1) if cuda_match else "unknown"

cpu_model = "Unknown CPU"
try:
    lscpu_out = run_cmd("lscpu")
    m = re.search(r"Model name:\s*(.+)", lscpu_out)
    if m:
        cpu_model = m.group(1).strip()
except Exception:
    cpu_model = platform.processor() or "Unknown CPU"

ram_gb = 0.0
try:
    with open("/proc/meminfo", "r") as f:
        for line in f:
            if line.startswith("MemTotal:"):
                kb = int(line.split()[1])
                ram_gb = round(kb / (1024 * 1024), 1)
                break
except Exception:
    pass

env_data = {
    "owner": "long",
    "cpu": cpu_model,
    "cores": os.cpu_count() or 2,
    "ram_gb": ram_gb,
    "os": f"{platform.system()} {platform.release()} (Google Colab)",
    "compiler": f"nvcc {cuda_ver} -O3 -std=c++14",
    "gpu": f"{gpu_name} ({vram_mb} MB, Driver {driver_ver}, CC {cap_raw})".strip(),
    "cuda": cuda_ver,
    "icarus": "N/A (GPU module)",
    "date": datetime.date.today().isoformat(),
}

with open("/tmp/env_long.json", "w", encoding="utf-8") as f:
    json.dump(env_data, f, indent=2, ensure_ascii=False)
    f.write("\n")

print("\n=== env_long.json đã chuẩn bị ===")
print(json.dumps(env_data, indent=2, ensure_ascii=False))

In [ ]:
import os
import shutil
import subprocess

REPO_URL = "https://github.com/ndh-mi1/projectKTMT.git"
WORK_DIR = "/content/projectKTMT"

if os.path.exists(os.path.join(WORK_DIR, ".git")):
    print("Repo đã tồn tại, đang cập nhật git pull...")
    os.chdir(WORK_DIR)
    subprocess.run(["git", "pull", "--rebase"], check=False)
else:
    print("Đang clone repo về", WORK_DIR)
    subprocess.run(["git", "clone", REPO_URL, WORK_DIR], check=True)
    os.chdir(WORK_DIR)

if os.path.exists("/tmp/env_long.json"):
    os.makedirs("results", exist_ok=True)
    shutil.copy("/tmp/env_long.json", "results/env_long.json")
    print("Đã lưu results/env_long.json")

print("Current directory:", os.getcwd())

In [ ]:
import os
import re
import subprocess

cap_raw = subprocess.check_output(
    "nvidia-smi --query-gpu=compute_cap --format=csv,noheader | head -n 1",
    shell=True,
    text=True,
).strip()

arch_flag = ""
if re.match(r"^[0-9]+\.[0-9]+$", cap_raw):
    sm = cap_raw.replace(".", "")
    arch_flag = f"-arch=sm_{sm}"
    print(f"Phát hiện Compute Capability {cap_raw} -> dùng cờ {arch_flag}")
else:
    print("Không đọc được compute_cap, dùng kiến trúc mặc định của nvcc.")

os.makedirs("gpu/bin", exist_ok=True)
compile_cmd = (
    f"nvcc -O3 -std=c++14 {arch_flag} -Igpu/src "
    "gpu/src/matmul_cuda.cu -o gpu/bin/bench_gpu -lcublas"
)
print("Compile command:", compile_cmd)
subprocess.run(compile_cmd, shell=True, check=True)
print("Biên dịch thành công: gpu/bin/bench_gpu")

In [ ]:
import os
import pandas as pd
import subprocess

demo_csv = "/tmp/demo_gpu.csv"
if os.path.exists(demo_csv):
    os.remove(demo_csv)

for n in [64, 128, 256]:
    cmd = f"./gpu/bin/bench_gpu --impl all --N {n} --repeats 5 --warmup 1 --out {demo_csv}"
    subprocess.run(cmd, shell=True, check=True)

df_demo = pd.read_csv(demo_csv)
print("\n=== Kết quả Demo nhanh ===")
display(df_demo)

assert (df_demo["correct"] == 1).all(), "Có phép tính không đạt ngưỡng sai số <= 1e-3!"
print("Tất cả test demo đều đạt correct == 1!")

for n, grp in df_demo.groupby("N"):
    t_naive = grp.loc[grp["impl"] == "cuda_naive", "kernel_ms"].values[0]
    t_cublas = grp.loc[grp["impl"] == "cublas", "kernel_ms"].values[0]
    print(f"N={n}: cuBLAS nhanh hơn cuda_naive {t_naive / t_cublas:.2f}x (kernel-only)")

In [ ]:
import os
import subprocess

out_csv = "results/gpu_bench.csv"
HEADER = "platform,impl,N,kernel_ms,h2d_ms,d2h_ms,e2e_ms,gflops_kernel,gflops_e2e,gpu_name,correct\n"

# Khởi tạo sạch file results/gpu_bench.csv với đúng header chuẩn METRICS.md
os.makedirs("results", exist_ok=True)
with open(out_csv, "w", encoding="utf-8") as f:
    f.write(HEADER)

cmd = f"./gpu/bin/bench_gpu --sweep --impl all --repeats 7 --warmup 2 --out {out_csv}"
print("Running:", cmd)
subprocess.run(cmd, shell=True, check=True)
print("Hoàn thành sweep toàn bộ dải N = 64 .. 4096!")

In [ ]:
import pandas as pd

df = pd.read_csv("results/gpu_bench.csv")
print(f"Tổng số dòng đo được: {len(df)} (kỳ vọng: 14 dòng cho 7 mức N x 2 impl)")
assert len(df) == 14, f"Số dòng không khớp: {len(df)}"
assert (df["correct"] == 1).all(), "Phát hiện kết quả sai số vượt ngưỡng!"

df["pcie_ms"] = df["h2d_ms"] + df["d2h_ms"]
df["pcie_overhead_%"] = 100.0 * df["pcie_ms"] / df["e2e_ms"]

# Tính speedup của cuBLAS so với cuda_naive theo từng N
naive_map = df[df["impl"] == "cuda_naive"].set_index("N")["kernel_ms"].to_dict()
df["speedup_vs_naive_kernel"] = df.apply(
    lambda r: naive_map[r["N"]] / r["kernel_ms"] if r["impl"] == "cublas" else 1.0,
    axis=1,
)

display(df)

print("\n=== Bảng Markdown chuẩn APA (copy vào Google Docs Mục 3.2) ===")
print("| impl | N | T_H2D (ms) | T_kernel (ms) | T_D2H (ms) | T_E2E (ms) | PCIe (%) | GFLOPS_kernel | GFLOPS_E2E | Speedup (kernel) | Correct |")
print("|---|---:|---:|---:|---:|---:|---:|---:|---:|---:|:---:|")
for _, r in df.iterrows():
    print(
        f"| {r['impl']} | {int(r['N'])} | {r['h2d_ms']:.4f} | {r['kernel_ms']:.4f} | "
        f"{r['d2h_ms']:.4f} | {r['e2e_ms']:.4f} | {r['pcie_overhead_%']:.1f}% | "
        f"{r['gflops_kernel']:.2f} | {r['gflops_e2e']:.2f} | {r['speedup_vs_naive_kernel']:.2f}x | {int(r['correct'])} |"
    )

In [ ]:
import os
import time
import numpy as np
import pandas as pd

gpu_df = pd.read_csv("results/gpu_bench.csv")

def load_cpu_best():
    frames = []
    for path in ["results/cpu_bench.csv", "results/cpu_opt_bench.csv"]:
        if os.path.exists(path):
            tmp = pd.read_csv(path)
            if len(tmp) > 0:
                frames.append(tmp)
    if not frames:
        return None
    all_cpu = pd.concat(frames, ignore_index=True)
    # Lấy thời gian nhỏ nhất (CPU tốt nhất) tại mỗi N
    best = all_cpu.groupby("N", as_index=False)["time_ms"].min()
    best = best.rename(columns={"time_ms": "cpu_best_ms"})
    return best

cpu_best_df = load_cpu_best()
cpu_source_label = "CPU tốt nhất của nhóm (results/cpu*.csv)"

if cpu_best_df is None:
    print(
        "[Thông báo] Chưa có dữ liệu trong results/cpu_bench.csv hoặc results/cpu_opt_bench.csv "
        "(Nam/Khương chưa push CSV). Đang đo tham chiếu NumPy (OpenBLAS) ngay trên CPU của Colab "
        "để minh họa cách tính điểm hòa vốn N*. Khi có CSV nhóm, chỉ cần chạy lại cell này!"
    )
    cpu_source_label = "Colab CPU NumPy/BLAS (minh họa tạm, thay bằng laptop CPU khi có CSV)"
    rows = []
    for n in [64, 128, 256, 512, 1024, 2048, 4096]:
        a = np.random.rand(n, n).astype(np.float32)
        b = np.random.rand(n, n).astype(np.float32)
        _ = a @ b  # warmup
        times = []
        reps = 5 if n <= 2048 else 3
        for _ in range(reps):
            t0 = time.perf_counter()
            _ = a @ b
            t1 = time.perf_counter()
            times.append((t1 - t0) * 1000.0)
        rows.append({"N": n, "cpu_best_ms": float(np.median(times))})
    cpu_best_df = pd.DataFrame(rows)

print(f"Nguồn dữ liệu CPU đối chứng: {cpu_source_label}")
merged_rows = []
for impl_name in ["cuda_naive", "cublas"]:
    sub = gpu_df[gpu_df["impl"] == impl_name].merge(cpu_best_df, on="N", how="inner").sort_values("N")
    n_star_kernel = sub.loc[sub["kernel_ms"] < sub["cpu_best_ms"], "N"].min()
    n_star_e2e = sub.loc[sub["e2e_ms"] < sub["cpu_best_ms"], "N"].min()
    print(f"\n--- {impl_name} vs CPU ---")
    print(f"  N*_kernel = {n_star_kernel if pd.notna(n_star_kernel) else 'Chưa vượt CPU trong dải đo'}")
    print(f"  N*_E2E    = {n_star_e2e if pd.notna(n_star_e2e) else 'Chưa vượt CPU trong dải đo'}")
    merged_rows.append(sub)

display(pd.concat(merged_rows, ignore_index=True)[["impl", "N", "cpu_best_ms", "kernel_ms", "e2e_ms", "gflops_kernel", "gflops_e2e"]])

In [ ]:
import os
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

os.makedirs("/content/gpu_plots", exist_ok=True)
gpu_df = pd.read_csv("results/gpu_bench.csv")
cublas_df = gpu_df[gpu_df["impl"] == "cublas"].sort_values("N")
naive_df = gpu_df[gpu_df["impl"] == "cuda_naive"].sort_values("N")

# --- Plot 1: Stacked percentage breakdown of T_H2D, T_kernel, T_D2H ---
fig, axes = plt.subplots(1, 2, figsize=(13, 5), sharey=True)
for ax, (impl_name, sub) in zip(axes, [("cuda_naive", naive_df), ("cublas", cublas_df)]):
    x = np.arange(len(sub))
    h2d_pct = 100.0 * sub["h2d_ms"] / sub["e2e_ms"]
    ker_pct = 100.0 * sub["kernel_ms"] / sub["e2e_ms"]
    d2h_pct = 100.0 * sub["d2h_ms"] / sub["e2e_ms"]
    ax.bar(x, h2d_pct, label="H2D (PCIe)", color="#3b82f6")
    ax.bar(x, ker_pct, bottom=h2d_pct, label="Kernel Compute", color="#10b981")
    ax.bar(x, d2h_pct, bottom=h2d_pct + ker_pct, label="D2H (PCIe)", color="#f59e0b")
    ax.set_xticks(x)
    ax.set_xticklabels([str(n) for n in sub["N"]])
    ax.set_xlabel("Kích thước ma trận N")
    ax.set_title(f"Phân rã tỷ trọng T_E2E — {impl_name}")
    ax.set_ylim(0, 100)
    ax.grid(axis="y", linestyle="--", alpha=0.4)

axes[0].set_ylabel("Tỷ trọng thời gian (%)")
axes[1].legend(loc="upper right")
plt.tight_layout()
plt.savefig("/content/gpu_plots/gpu_time_breakdown.png", dpi=300)
plt.show()

# --- Plot 2: Log-log execution time (GPU kernel & E2E vs CPU best) ---
fig, ax = plt.subplots(figsize=(8.5, 5.5))
ax.plot(naive_df["N"], naive_df["kernel_ms"], "o--", label="cuda_naive (Kernel)")
ax.plot(naive_df["N"], naive_df["e2e_ms"], "o-", label="cuda_naive (E2E)")
ax.plot(cublas_df["N"], cublas_df["kernel_ms"], "s--", label="cuBLAS (Kernel)")
ax.plot(cublas_df["N"], cublas_df["e2e_ms"], "s-", label="cuBLAS (E2E)")

if "cpu_best_df" in globals() and cpu_best_df is not None:
    ax.plot(cpu_best_df["N"], cpu_best_df["cpu_best_ms"], "^-", linewidth=2, color="black", label=cpu_source_label)
    m_cub = cublas_df.merge(cpu_best_df, on="N", how="inner")
    nk = m_cub.loc[m_cub["kernel_ms"] < m_cub["cpu_best_ms"], "N"].min()
    ne = m_cub.loc[m_cub["e2e_ms"] < m_cub["cpu_best_ms"], "N"].min()
    if pd.notna(nk):
        yk = float(m_cub.loc[m_cub["N"] == nk, "kernel_ms"].values[0])
        ax.annotate(f"N*_kernel={int(nk)}", xy=(nk, yk), xytext=(nk * 0.7, yk * 0.35),
                    arrowprops=dict(arrowstyle="->", color="green"), fontsize=9, fontweight="bold")
    if pd.notna(ne):
        ye = float(m_cub.loc[m_cub["N"] == ne, "e2e_ms"].values[0])
        ax.annotate(f"N*_E2E={int(ne)}", xy=(ne, ye), xytext=(ne * 0.7, ye * 2.2),
                    arrowprops=dict(arrowstyle="->", color="red"), fontsize=9, fontweight="bold")

ax.set_xscale("log", base=2)
ax.set_yscale("log", base=10)
ax.set_xlabel("Kích thước ma trận N (log2)")
ax.set_ylabel("Thời gian thực thi ms (log10)")
ax.set_title("So sánh thời gian thực thi GPU (Kernel vs E2E) và Điểm hòa vốn N*")
ax.grid(True, which="both", linestyle="--", alpha=0.4)
ax.legend()
plt.tight_layout()
plt.savefig("/content/gpu_plots/gpu_vs_cpu_breakeven.png", dpi=300)
plt.show()

# --- Plot 3: Throughput scaling (GFLOPS kernel vs E2E) + Arithmetic Intensity I_PCIe ---
fig, ax1 = plt.subplots(figsize=(8.5, 5.2))
ax1.plot(naive_df["N"], naive_df["gflops_kernel"], "o--", label="cuda_naive GFLOPS (Kernel)")
ax1.plot(naive_df["N"], naive_df["gflops_e2e"], "o-", label="cuda_naive GFLOPS (E2E)")
ax1.plot(cublas_df["N"], cublas_df["gflops_kernel"], "s--", label="cuBLAS GFLOPS (Kernel)")
ax1.plot(cublas_df["N"], cublas_df["gflops_e2e"], "s-", label="cuBLAS GFLOPS (E2E)")
ax1.set_xscale("log", base=2)
ax1.set_xlabel("Kích thước ma trận N (log2)")
ax1.set_ylabel("Thông lượng (GFLOPS)")
ax1.set_title("Thông lượng GFLOPS (Kernel vs E2E) và Cường độ số học PCIe")
ax1.grid(True, which="both", linestyle="--", alpha=0.4)

ax2 = ax1.twinx()
n_vals = cublas_df["N"].values
i_pcie = n_vals / 6.0
ax2.plot(n_vals, i_pcie, "d:", color="purple", alpha=0.75, label="I_PCIe = N/6 (FLOPs/Byte)")
ax2.set_ylabel("Cường độ số học I_PCIe (FLOPs/Byte)", color="purple")

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc="upper left", fontsize=8.5)
plt.tight_layout()
plt.savefig("/content/gpu_plots/gpu_gflops_scaling.png", dpi=300)
plt.show()
print("Đã xuất 3 biểu đồ tại /content/gpu_plots/")

In [ ]:
try:
    from google.colab import files
    files.download("results/gpu_bench.csv")
    files.download("results/env_long.json")
    print("Đã gửi tải xuống results/gpu_bench.csv và results/env_long.json.")
except ImportError:
    print("Không ở trong môi trường Google Colab; các file đã sẵn sàng tại:")
    print("  - results/gpu_bench.csv")
    print("  - results/env_long.json")

print("\nSau khi tải về máy local, đặt vào thư mục results/ của repo và chạy:")
print("  git pull")
print("  git add gpu/ results/gpu_bench.csv results/env_long.json")
print('  git commit -m "[Long] cap nhat CUDA kernel, Colab notebook, gpu_bench.csv va env_long.json"')
print("  git push")